# Phase 4 — ML Baseline: Random Forest Evaluation & Verification

**Status:** ✅ Completed (pipeline v1.0)

This notebook is a **read-only validation layer** over `ml_service/src/train_rf.py`.
It inspects `artifacts/rf_metrics_v1.json` and presents the empirical baseline evaluation:
- Binary Intrusion Detection (TPR/Detection Rate, FPR, Precision, Recall, F1, ROC-AUC)
- Multi-Class Attack Categorization across all 15 CIC-IDS2017 classes
- Rare-Class Statistical Constraint Analysis (Heartbleed, Infiltration, SQL Injection)
- Operational Latency and Peak Memory Telemetry

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import numpy as np

root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from config.paths import Paths

with open(Paths.ARTIFACTS_DIR / 'rf_metrics_v1.json', 'r', encoding='utf-8') as f:
    metrics = json.load(f)

hp = metrics['hyperparameters']
print(f"Phase:             {metrics['phase']}")
print(f"Model:             Random Forest Baseline (LightGBM boosting_type='rf')")
print(f"Evaluation Time:   {metrics['evaluation_timestamp']}")
print(f"Test Sample Count: {metrics['test_sample_count']:,}")
print(f"Training Time:     {hp['train_time_seconds']:.2f} s")
print(f"Inference Latency: {metrics['inference_latency_us_per_sample']:.2f} μs/sample")

## 1. Binary Intrusion Detection Benchmark (Attack vs. Benign)

In [ ]:
bm = metrics['binary_intrusion_detection']
cm = bm['confusion_matrix']

bin_rows = [
    {'Metric': 'Accuracy', 'Score': f"{bm['accuracy']*100:.3f}%"},
    {'Metric': 'Detection Rate (Recall / TPR)', 'Score': f"{bm['recall_tpr']*100:.3f}%"},
    {'Metric': 'False Positive Rate (FPR)', 'Score': f"{bm['false_positive_rate_fpr']*100:.4f}%"},
    {'Metric': 'Precision', 'Score': f"{bm['precision']*100:.3f}%"},
    {'Metric': 'F1-Score', 'Score': f"{bm['f1_score']:.5f}"},
    {'Metric': 'ROC-AUC', 'Score': f"{bm['roc_auc']:.5f}"},
]
pd.DataFrame(bin_rows)

### Binary Confusion Matrix
- **True Negatives (BENIGN correctly passed):** `411,610`
- **False Positives (BENIGN false alarms):** `7,402`
- **False Negatives (ATTACK missed):** `51` (out of 85,148 total attacks)
- **True Positives (ATTACK detected):** `85,097`

In [ ]:
cm_matrix = pd.DataFrame(
    [[cm['true_negatives'], cm['false_positives']],
     [cm['false_negatives'], cm['true_positives']]],
    index=['Actual BENIGN', 'Actual ATTACK'],
    columns=['Predicted BENIGN', 'Predicted ATTACK']
)
cm_matrix.style.format('{:,}')

## 2. 15-Class Attack Categorization Performance

In [ ]:
gm = metrics['global_multiclass']
print(f"Overall Accuracy:        {gm['accuracy']*100:.3f}%")
print(f"Balanced Accuracy:       {gm['balanced_accuracy']*100:.3f}%")
print(f"Weighted F1-Score:       {gm['weighted_f1']:.5f}")
print(f"Macro F1-Score:          {gm['macro_f1']:.5f}")
print(f"ROC-AUC (OVR Weighted):  {gm['roc_auc_ovr_weighted']:.5f}")

### Detailed Per-Class Breakdown

In [ ]:
per_class = metrics['per_class']
pc_rows = []
for cls, data in per_class.items():
    stat_note = '⚠️ Low Support (<30)' if data['low_support_warning'] else '✅ Robust'
    pc_rows.append({
        'Attack Category': cls,
        'Test Support': data['support'],
        'Precision': data['precision'],
        'Recall': data['recall'],
        'F1-Score': data['f1_score'],
        'Statistical Quality': stat_note
    })

pd.DataFrame(pc_rows).style.format({
    'Test Support': '{:,}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-Score': '{:.4f}'
})

## 3. Hardware Telemetry & Memory Profile

In [ ]:
mem = metrics['memory_telemetry']
mem_rows = [{'Checkpoint': k.replace('_', ' ').title(), 'Resident RSS': f"{v:.2f} MB"} for k, v in mem.items()]
pd.DataFrame(mem_rows)